In [1]:
# 03_modeling_airbnb.ipynb - GÜNCELLENMİŞ
# ===============================
# 1. Bibliotheken
# ===============================
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

print("🧠 Modellvergleich gestartet...")

# ===============================
# 2. Daten laden
# ===============================
data_path = os.path.join(os.path.dirname(os.getcwd()), "data", "processed")
X_train = pd.read_csv(os.path.join(data_path, "X_train.csv"))
X_test = pd.read_csv(os.path.join(data_path, "X_test.csv"))
y_train = pd.read_csv(os.path.join(data_path, "y_train.csv"))
y_test = pd.read_csv(os.path.join(data_path, "y_test.csv"))

print(f"📊 Daten geladen: X_train {X_train.shape}, X_test {X_test.shape}")

# ===============================
# 3. Datenüberprüfung
# ===============================
print("\n🔍 Datenüberprüfung:")
print("X_train Datentypen:", X_train.dtypes.value_counts())
print("X_test Datentypen:", X_test.dtypes.value_counts())

# Sicherstellen, dass alle Daten numerisch sind
for col in X_train.columns:
    if X_train[col].dtype == 'object':
        print(f"❌ Konvertiere {col} zu numerisch...")
        X_train[col] = pd.to_numeric(X_train[col], errors='coerce')
        X_test[col] = pd.to_numeric(X_test[col], errors='coerce')

X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

# y-Daten als Array konvertieren
y_train = y_train.values.ravel()
y_test = y_test.values.ravel()

# ===============================
# 4. Modelle definieren
# ===============================
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=50, random_state=42)
}

# ===============================
# 5. Modelltraining und Bewertung
# ===============================
results = []

for name, model in models.items():
    print(f"\n🔧 Trainiere {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        "Modell": name,
        "MAE": mae,
        "RMSE": rmse,
        "R²": r2
    })

results_df = pd.DataFrame(results).round(3)
print("\n📊 Modellvergleich:")
print(results_df)

# ===============================
# 6. Bestes Modell speichern
# ===============================
best_model_name = results_df.sort_values(by="RMSE").iloc[0]["Modell"]
best_model = models[best_model_name]

models_path = os.path.join(os.path.dirname(os.getcwd()), "models")
os.makedirs(models_path, exist_ok=True)
model_path = os.path.join(models_path, "best_model.pkl")
joblib.dump(best_model, model_path)

print(f"\n✅ Bestes Modell ({best_model_name}) wurde gespeichert unter: {model_path}")


🧠 Modellvergleich gestartet...
📊 Daten geladen: X_train (38868, 6), X_test (9718, 6)

🔍 Datenüberprüfung:
X_train Datentypen: int64      4
float64    2
Name: count, dtype: int64
X_test Datentypen: int64      4
float64    2
Name: count, dtype: int64

🔧 Trainiere Linear Regression...

🔧 Trainiere Random Forest...



🔧 Trainiere Gradient Boosting...



📊 Modellvergleich:
              Modell     MAE    RMSE     R²
0  Linear Regression  58.594  97.031  0.267
1      Random Forest  52.390  91.123  0.353
2  Gradient Boosting  52.218  91.172  0.352

✅ Bestes Modell (Random Forest) wurde gespeichert unter: C:\Users\okans\OneDrive\Pictures\Masaüstü\airbnb-data-analysis\models\best_model.pkl
